In [1]:

# ============================================================
# EduInsight AI
# Phase 3: Data Cleaning
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")

# 1. PATHS

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. FIND DATA FILES

supported_formats = [".csv", ".xlsx", ".xls", ".json"]

data_files = [
    f for f in RAW_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in supported_formats
    and f.name != "dataset_summary.csv"
]

if not data_files:
    raise FileNotFoundError(
        "No dataset files found in data/raw. "
        "Place and extract the dataset first."
    )

# 3. LOAD DATA

datasets = []

for file in data_files:
    try:
        if file.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(file, encoding="utf-8")
            except UnicodeDecodeError:
                df = pd.read_csv(file, encoding="latin-1")

        elif file.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(file)

        elif file.suffix.lower() == ".json":
            try:
                df = pd.read_json(file)
            except ValueError:
                with open(file, "r", encoding="utf-8") as f:
                    df = pd.json_normalize(json.load(f))

        if not df.empty:
            datasets.append((file.stem, df))

    except Exception as error:
        print(f"Skipped {file.name}: {error}")

if not datasets:
    raise ValueError("No readable datasets were found.")

# 4. COMBINE COMPATIBLE DATASETS

base_name, data = datasets[0]

for name, other_df in datasets[1:]:
    if set(data.columns) == set(other_df.columns):
        other_df = other_df[data.columns]
        data = pd.concat([data, other_df], ignore_index=True)

print("Original dataset shape:", data.shape)

# 5. STANDARDIZE COLUMN NAMES

data.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in data.columns
]

# 6. REMOVE EMPTY ROWS AND COLUMNS

data = data.dropna(how="all")
data = data.dropna(axis=1, how="all")

# 7. REMOVE DUPLICATES

duplicate_count = data.duplicated().sum()
data = data.drop_duplicates()

# 8. CLEAN TEXT COLUMNS

for column in data.select_dtypes(include=["object"]).columns:
    data[column] = data[column].apply(
        lambda value: value.strip()
        if isinstance(value, str) else value
    )

# 9. REMOVE COMPLETELY EMPTY TEXT VALUES

for column in data.select_dtypes(include=["object"]).columns:
    data[column] = data[column].replace(
        r"^\s*$", np.nan, regex=True
    )

# 10. CHECK MISSING VALUES

missing_report = pd.DataFrame({
    "missing_count": data.isnull().sum(),
    "missing_percentage": (
        data.isnull().mean() * 100
    ).round(2)
})

print("\nMissing values report:")
display(missing_report)

# 11. FINAL DATASET INFORMATION

print("\nCleaned dataset shape:", data.shape)
print("Duplicates removed:", duplicate_count)

print("\nData types:")
print(data.dtypes)

print("\nFirst 5 rows:")
display(data.head())

# 12. SAVE CLEANED DATA

output_path = PROCESSED_DIR / "cleaned_education_feedback.csv"

data.to_csv(output_path, index=False, encoding="utf-8-sig")

# 13. SAVE CLEANING REPORT

report = {
    "original_rows": int(sum(len(df) for _, df in datasets)),
    "cleaned_rows": int(len(data)),
    "columns": int(len(data.columns)),
    "duplicates_removed": int(duplicate_count),
    "missing_values": int(data.isnull().sum().sum())
}

with open(
    PROCESSED_DIR / "cleaning_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(report, file, indent=4, ensure_ascii=False)

print("\nCleaning completed!")
print("Cleaned dataset:", output_path)
print("Cleaning report saved.")


FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'c:\\Users\\User\\Desktop\\EduInsight-AI\\data\\processed'

In [2]:

from pathlib import Path
import zipfile
import shutil

# PROJECT DIRECTORY
PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")

# RAW DIRECTORY
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# SEARCH FOR ZIP FILE ON DESKTOP
desktop = Path.home() / "Desktop"

zip_files = list(desktop.rglob("EduRABSA_Dataset.zip"))

if not zip_files:
    raise FileNotFoundError(
        "EduRABSA_Dataset.zip was not found on Desktop."
    )

# SELECT DATASET
zip_path = zip_files[0]

print("Dataset found:", zip_path)

# COPY ZIP TO RAW
destination = RAW_DIR / zip_path.name

if zip_path.resolve() != destination.resolve():
    shutil.copy2(zip_path, destination)

# EXTRACT DATASET
with zipfile.ZipFile(destination, "r") as archive:
    archive.extractall(RAW_DIR)

print("\nDataset successfully copied and extracted!")
print("Raw directory:", RAW_DIR)

# SHOW EXTRACTED FILES
for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(file.relative_to(RAW_DIR))


FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\User\\Desktop\\EduInsight-AI\\data\\raw'

In [3]:

from pathlib import Path
import zipfile
import shutil

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
RAW_DIR = PROJECT_DIR / "data" / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

desktop = Path.home() / "Desktop"
zip_files = list(desktop.rglob("EduRABSA_Dataset.zip"))

if not zip_files:
    raise FileNotFoundError("ZIP file not found on Desktop.")

zip_path = zip_files[0]
destination = RAW_DIR / zip_path.name

if zip_path.resolve() != destination.resolve():
    if not destination.exists():
        shutil.copy2(zip_path, destination)

# Extract without overwriting existing files
with zipfile.ZipFile(destination, "r") as archive:
    for member in archive.infolist():
        target = RAW_DIR / member.filename

        if not target.exists():
            archive.extract(member, RAW_DIR)

print("Dataset extraction completed!")

for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(file.relative_to(RAW_DIR))


FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\User\\Desktop\\EduInsight-AI\\data\\raw'

In [4]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
RAW_DIR = PROJECT_DIR / "data" / "raw"

print("raw mövcuddur:", RAW_DIR.exists())
print("Qovluqdur:", RAW_DIR.is_dir())
print("Fayldır:", RAW_DIR.is_file())

raw mövcuddur: True
Qovluqdur: False
Fayldır: True


In [5]:

from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")

RAW_PATH = PROJECT_DIR / "data" / "raw"

# Rename the existing file
if RAW_PATH.is_file():
    RAW_PATH.rename(
        PROJECT_DIR / "data" / "raw_old"
    )

# Create the correct folder
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Raw directory created:", RAW_DIR.is_dir())
print("Path:", RAW_DIR)


Raw directory created: True
Path: C:\Users\User\Desktop\EduInsight-AI\data\raw


In [6]:

from pathlib import Path
import zipfile
import shutil

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
RAW_DIR = PROJECT_DIR / "data" / "raw"

# Find ZIP on Desktop
desktop = Path.home() / "Desktop"
zip_files = list(desktop.rglob("EduRABSA_Dataset.zip"))

if not zip_files:
    raise FileNotFoundError("EduRABSA_Dataset.zip not found.")

zip_path = zip_files[0]
destination = RAW_DIR / zip_path.name

# Copy ZIP
if zip_path.resolve() != destination.resolve():
    if not destination.exists():
        shutil.copy2(zip_path, destination)

# Extract ZIP
with zipfile.ZipFile(destination, "r") as archive:
    archive.extractall(RAW_DIR)

print("Dataset extracted successfully!")

# List extracted files
for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(file.relative_to(RAW_DIR))


FileNotFoundError: EduRABSA_Dataset.zip not found.

In [7]:

from pathlib import Path

search_locations = [
    Path.home() / "Desktop",
    Path.home() / "Downloads",
    Path.home() / "Documents"
]

found_files = []

for location in search_locations:
    if location.exists():
        for file in location.rglob("*"):
            if file.is_file() and "edurabsa" in file.name.lower():
                found_files.append(file)

print("SEARCH RESULTS:")

for file in found_files:
    print(file)

if not found_files:
    print("Dataset ZIP was not found.")


SEARCH RESULTS:
C:\Users\User\Downloads\EduRABSA_Dataset.zip


In [8]:

from pathlib import Path
import zipfile
import shutil

# Project paths
PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
RAW_DIR = PROJECT_DIR / "data" / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

# Dataset location
ZIP_PATH = Path(
    r"C:\Users\User\Downloads\EduRABSA_Dataset.zip"
)

# Copy dataset into raw folder
DESTINATION = RAW_DIR / ZIP_PATH.name

if not DESTINATION.exists():
    shutil.copy2(ZIP_PATH, DESTINATION)

# Extract dataset
with zipfile.ZipFile(DESTINATION, "r") as archive:
    archive.extractall(RAW_DIR)

# Display extracted files
print("=" * 50)
print("DATASET EXTRACTED SUCCESSFULLY")
print("=" * 50)

for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(file.relative_to(RAW_DIR))


DATASET EXTRACTED SUCCESSFULLY
EduRABSA_Dataset.zip
EduRABSA_Dataset\LICENSE-CC-BY.txt
EduRABSA_Dataset\2_annotated_dataset_files\train_test_split_info.md
EduRABSA_Dataset\2_annotated_dataset_files\Annotation_raw_output\DEV_TEST_Merged_ASQE_N2500.json
EduRABSA_Dataset\2_annotated_dataset_files\Annotation_raw_output\Train_validation_Merged_ASQE_N4000.json
EduRABSA_Dataset\2_annotated_dataset_files\pyabsa_dataset_format\AEOE\DEV_TEST_Merged_AEOE_2500.txt
EduRABSA_Dataset\2_annotated_dataset_files\pyabsa_dataset_format\AEOE\Train_val_Merged_AEOE_4000.txt
EduRABSA_Dataset\2_annotated_dataset_files\pyabsa_dataset_format\APC\DEV_TEST_Merged_APC_2500.txt
EduRABSA_Dataset\2_annotated_dataset_files\pyabsa_dataset_format\APC\Train_val_Merged_APC_4000.txt
EduRABSA_Dataset\2_annotated_dataset_files\pyabsa_dataset_format\ASQE\DEV_TEST_Merged_ASQE_N2500.jsonl
EduRABSA_Dataset\2_annotated_dataset_files\pyabsa_dataset_format\ASQE\Train_validation_Merged_ASQE_N4000.jsonl
EduRABSA_Dataset\2_annotated_da